In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from churn import config


In [2]:
df=pd.read_parquet(config.CLEAN_DATA_FILE)
X = df.drop(columns=[config.TARGET])
y = df[config.TARGET]

Train-test-split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [4]:
print(f"X_train.shape: {X_train.shape}, X_test.shape: {X_test.shape}")
print(f"churn rate  train={y_train.mean():.4f}  test={y_test.mean():.4f}")

X_train.shape: (5634, 20), X_test.shape: (1409, 20)
churn rate  train=0.2654  test=0.2654


In [5]:
X_train.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
3738,4950-BDEUX,Male,No,No,No,35,No,No phone service,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65
3151,7993-NQLJE,Male,No,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55
4860,7321-ZNSLA,Male,No,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35
3867,4922-CVPDX,Female,No,Yes,No,26,Yes,No,DSL,No,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.70
3810,2903-YYTBW,Male,No,Yes,Yes,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55


#### [Sklearn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)

A sequence of data transformers with an optional final predictor.

`Pipeline` allows you to sequentially apply a list of transformers to preprocess the data and, if desired, conclude the sequence with a final predictor for predictive modeling.

Intermediate steps of the pipeline must be transformers, that is, they must implement `fit` and `transform` methods. The final estimator only needs to implement `fit`. The transformers in the pipeline can be cached using memory argument.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer

from churn.features import collapse_no_internet, build_preprocessor, add_features

In [7]:
def build_pipeline(model, model_type, num_cols=None, cat_cols=None):
    """Raw rows in → churn prediction out. model_type: "lr" or "tree" (picks the preprocessor)."""
    return Pipeline([
        ("collapse", FunctionTransformer(collapse_no_internet)),
        ("features", FunctionTransformer(add_features)),
        ("prep", build_preprocessor(model_type, num_cols, cat_cols)),
        ("model", model),
    ])

In [8]:
pipe_lr = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr").fit(X_train, y_train)

In [9]:
pipe_lr[:-1].transform(X_train.head())

,num__tenure,num__MonthlyCharges,cat__SeniorCitizen_Yes,cat__Partner_Yes,cat__Dependents_Yes,cat__MultipleLines_No,cat__MultipleLines_No phone service,cat__MultipleLines_Yes,cat__InternetService_DSL,cat__InternetService_Fiber optic,...,cat__TechSupport_Yes,cat__StreamingTV_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check
3738,0.102371,-0.521976,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3151,-0.711743,0.337478,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4860,-0.793155,-0.809013,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3867,-0.263980,0.284384,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
3810,-1.281624,-0.676279,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [10]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [11]:
# Define metrics relevant for imbalanced data
scoring_metrics = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [12]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_lr = cross_validate(pipe_lr, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs = -1)
cv_lr_df = pd.DataFrame(cv_lr)
cv_lr_df

,fit_time,score_time,test_f1,test_precision,test_recall,test_roc_auc,test_pr_auc
0,0.154875,0.097008,0.585185,0.655602,0.528428,0.846764,0.643585
1,0.177492,0.110238,0.553114,0.611336,0.505017,0.825594,0.638309
2,0.166838,0.093261,0.583643,0.656904,0.525084,0.840240,0.680587
3,0.163692,0.089797,0.633218,0.655914,0.612040,0.860455,0.674025
4,0.155086,0.078620,0.596958,0.691630,0.525084,0.850829,0.661312


#### Notes 1: how to read `cv_lr_df` (Logistic Regression, 5-fold CV)

**The table**
- **1 row = 1 fold.** Each fold trains the whole Pipeline on 4/5 of `X_train` and scores the remaining 1/5 (≈ 1,056 customers).
- **`test_*` = the validation fold**, *not* `X_test`. That's sklearn's naming; the real test set stays untouched.
- `fit_time` / `score_time`: seconds to train / to predict + score. Useful later to compare slow vs fast models.

**`.agg(["mean", "std"]).T`**
- **mean** = the score to expect on new customers. Use it to **compare models**.
- **std** = how much the score moves between folds, i.e. how **stable** (trustworthy) the mean is.
- Rule of thumb: if two models' means differ by **less than about one std**, the difference may just be noise from how the folds were split. Don't call a winner on it.

**Is a small std good?** Yes. It means the score doesn't depend on which customers ended up in which fold. For ≈ 1,000 rows per fold:

| std | Reading |
|---|---|
| < 0.02 | Very stable |
| 0.02 – 0.05 | Normal for this data size (our P, R, F1, PR-AUC ≈ 0.03–0.05) |
| > 0.05 | Unstable: check the data or the model before trusting the mean |

**What counts as a good score?**

| Metric | Random model | Our LR | Good on this dataset | Warning sign |
|---|---|---|---|---|
| ROC-AUC | 0.50 | **0.85 ± 0.02** | 0.83 – 0.86 (typical tuned models) | > 0.90 → suspect leakage |
| PR-AUC | 0.265 (= churn rate) | **0.66 ± 0.05** | 0.60 – 0.70 | Near 0.265 → model learned nothing |
| Precision / Recall / F1 | depends on threshold | 0.68 / 0.53 / 0.59 | No fixed range: they change with the threshold (phase 6) | n/a |

ROC-AUC and PR-AUC don't use a cut-off, so they get "good ranges". Precision, recall and F1 here use the default 0.5 cut-off, so they are only a first look.


In [13]:
cv_lr_df.agg(["mean", "std"]).T   # .T turns it so each metric is a row


,mean,std
fit_time,0.163597,0.009382
score_time,0.093785,0.011481
test_f1,0.590424,0.028894
test_precision,0.654277,0.028507
test_recall,0.539130,0.041799
test_roc_auc,0.844776,0.012990
test_pr_auc,0.659564,0.018448


### Notes 2: what the mean scores mean for the business

**The confusion matrix words** (positive = churn = 1):

| | Model says **churn** (predicted 1, flagged) | Model says **stays** (predicted 0) |
|---|---|---|
| **Really churns** (actual 1) | **TP**: caught, gets an offer ✅ | **FN**: missed, leaves → *lost revenue* (expensive) |
| **Really stays** (actual 0) | **FP**: false alarm → *wasted offer* (cheaper) | **TN**: correctly left alone |

**One validation fold in numbers** (≈ 1,056 customers, ≈ 280 real churners), with the mean scores at the 0.5 cut-off:

| Metric | Formula | Mean | In plain words | In one fold |
|---|---|---|---|---|
| **Precision** | TP / (TP + FP) | 0.68 | Of the customers we **flag**, 68% really churn | ≈ 218 flagged → 148 real churners, **70 wasted offers** |
| **Recall** | TP / (TP + FN) | 0.53 | Of all real **churners**, we catch 53% | 148 caught, **132 missed** (lost revenue) |
| **F1** | 2·P·R / (P + R) | 0.59 | One number that balances P and R; low if either one is low | n/a |
| **ROC-AUC** | P(random churner scored higher than random stayer) | 0.85 | The model **ranks** churners above stayers 85% of the time | Uses probabilities, not the cut-off |
| **PR-AUC** | Precision averaged over all recall levels | 0.66 | How good the flagged list is at every list length. Random = 0.265 | ≈ 2.5× better than random |

**Business reading**
- **Recall gets extra weight:** a missed churner (FN) costs more than a wasted offer (FP), but precision keeps the budget and team workload in check. Phase 6 balances the two by cost.
- **Precision protects the budget and the team's time.** It says how many offers are wasted, and the number flagged must fit the retention team's capacity.
- **Precision, recall and F1 change with the threshold.** Lowering the cut-off from 0.5 flags more customers: recall ↑, precision ↓. Phase 6 picks the threshold from the cost of FN vs FP. So, for now, **don't compare models on these three**.
- **ROC-AUC and PR-AUC don't depend on the threshold.** They measure how well the model *orders* customers by risk, which is what the retention team uses ("call the top N"). **Models are ranked on these two in phase 5.**
- **Accuracy is left out on purpose.** Predicting "nobody churns" already scores 73.5% but catches 0 churners.

**Why recall gets more weight than precision: the two mistakes cost different amounts**

| Metric | The mistake it counts | What that mistake costs |
|---|---|---|
| **Recall** = TP / (TP + **FN**) | **FN**: a churner we didn't flag | They leave. We lose their future monthly revenue. |
| **Precision** = TP / (TP + **FP**) | **FP**: a loyal customer we flagged | We give a discount to someone who would have stayed. We lose the cost of one offer. |

> ⚠️ **The dollar figures below are MADE-UP numbers**, only to show the idea. The dataset has no cost data, and no spec gives these values: phase 1 defines only the *formulas*, and the real numbers are chosen and justified in **phase 6, step T1**. Only the ~$75/month is loosely based on the data (≈ average `MonthlyCharges` of churners); **12 months, 30% and $50 are invented.**

- **FN (missed churner):** lost revenue ≈ `MonthlyCharges` × months they'd have stayed × chance the offer would have kept them.
  - Made-up example: $75/month × 12 months × 30% success ≈ **$270**.
- **FP (wasted offer):** one discount; made-up example **$50**.

With these made-up numbers, **one missed churner costs about 5× more than one wasted offer**. Trading one FN for a few FPs is then still a profit, which is why the business leans towards **catching more churners (recall)**, even if a few more loyal customers get a discount (lower precision). Step T7 checks how much the threshold moves if the real ratio is 3:1 or 10:1 instead.


### Notes 3: ROC-AUC and PR-AUC (the two metrics we rank models on)

Both start from the model's **churn probability** (`predict_proba`), not from a yes/no prediction. Picture the model sorting all customers into a **risk list**, highest risk at the top. Every possible threshold is a cut-off point in that list: everyone above it gets an offer. Both curves show what happens at **every** cut-off, so they judge the **ranking**, not one threshold.

---

#### 1. Full forms and definitions

| | **ROC-AUC** | **PR-AUC** (sklearn: `average_precision`) |
|---|---|---|
| Full form | **R**eceiver **O**perating **C**haracteristic, **A**rea **U**nder the **C**urve | **P**recision–**R**ecall, **A**rea **U**nder the **C**urve |
| Plots | **TPR** (y) vs **FPR** (x) | **Precision** (y) vs **Recall** (x) |
| Formulas | TPR = TP / (TP + FN) = **recall**<br>FPR = FP / (FP + TN) | Precision = TP / (TP + FP)<br>Recall = TP / (TP + FN) |
| Definition | Probability that a **random churner** gets a **higher** risk score than a **random loyal customer** | Average precision over all recall levels: how "clean" the call list stays as we go further down it |
| Random model | **0.50** | **0.265** (= the churn rate) |
| Perfect model | 1.0 | 1.0 |
| Our LR (CV) | **0.85 ± 0.02** | **0.66 ± 0.05** |

**In Telco business words**
- **TPR (recall):** share of churners we reach with an offer. *Revenue saved.*
- **FPR:** share of **loyal** customers who get an offer they didn't need. *Wasted discounts.*
- **Precision:** share of the **call list** that really churns. *How many of the team's calls are worth making.*
- **ROC-AUC 0.85:** pick one customer who churned and one who stayed; 85% of the time the model puts the churner higher on the risk list.
- **PR-AUC 0.66:** as the retention team works down the list, on average about 2 of every 3 calls reach a real churner (random calling: about 1 in 4).

---

#### 2. What the charts look like

**ROC curve**
```
TPR   1.0 |        ____-----‾‾‾‾
(recall)  |    _-‾‾          .·
          |  /'           .·      ← diagonal = random guessing (AUC 0.5)
          | /   ← ours  .·
          |/  (AUC .85).·
      0.0 +·------------------
          0.0      FPR       1.0
```
- Each point on the curve is **one threshold**. Bottom-left: threshold 1.0, nobody flagged. Top-right: threshold 0, everyone flagged.
- **Closer to the top-left corner = better:** we catch many churners (high TPR) while bothering few loyal customers (low FPR).
- Our 0.5 cut-off is one point: about (FPR 0.09, TPR 0.53) in a fold. 70 of ≈ 776 loyal customers flagged, 148 of ≈ 280 churners caught.  
- (FPR 0.09, TPR 0.53) => It catches about 53% of churners while bothering about 9% of loyal customers.

**PR curve**
```
Precision 1.0 |‾‾\_
              |    ‾\__          ← ours (AUC .66)
              |        ‾‾\__
              |             ‾\_
        0.265 |- - - - - - - - -‾‾  ← random = churn rate (flat line)
              +-------------------
              0.0     Recall     1.0
```
- Read it **left to right as the call list getting longer**. On the left, only the top-risk customers are called and precision is high. Calling more customers raises recall, but more of the calls are wasted, so precision falls.
- **Higher and flatter = better:** the list stays "clean" for longer.
- At recall 1.0 (call everyone) precision drops to 0.265, because then the list is just the whole customer base.

**Why look at both?** Churners are the minority (26.5%). On the ROC chart, 70 wasted offers look small (FPR = 70 / 776 ≈ 9%), because the denominator is all loyal customers. On the PR chart, the same 70 are 70 of the 218 calls (≈ 32% wasted), which is what the retention team actually feels. **ROC-AUC = overall ranking skill; PR-AUC = quality of the call list.**



#### 3. Using them to compare models in CV and pick the best

**a) The numbers (what the comparison table uses)**
1. For each model, run `cross_validate` and take `test_roc_auc` and `test_pr_auc` as **mean ± std** over the 5 folds.
2. **Rank by mean ROC-AUC**; use **PR-AUC as the tie-breaker** (the spec's rule).
3. **Gap smaller than about one std → treat it as a tie.** Prefer the simpler, more explainable model (e.g. LR over XGBoost). If LR is within ~0.01 of the best boosted model, say so openly.
4. **Check overfitting:** with `return_train_score=True`, train ROC-AUC − CV ROC-AUC should be < ~0.05 (leakage check L6).
5. **Sanity check:** CV ROC-AUC > 0.90 on this dataset → suspect leakage, not a great model.

**b) The curves (the picture behind the numbers)**
- To draw CV curves without touching `X_test`, get **out-of-fold probabilities** on the training set: `cross_val_predict(pipe, X_train, y_train, cv=skf, method="predict_proba")[:, 1]`. Each customer is scored by a model that never saw them.
- Plot every model's ROC curve on one chart and every PR curve on another (`RocCurveDisplay.from_predictions`, `PrecisionRecallDisplay.from_predictions`).
- **One curve above another everywhere** → that model is better at every threshold. Easy choice.
- **Curves cross** → look at the region the business will use. If the team can only call the top ~20% of customers, compare the curves at **low FPR / low recall** (the left side), not the whole area.

**c) Business link**
- AUC picks the model that **ranks** churners best. That's the right job for phase 5, because the threshold isn't chosen yet.
- In **phase 6**, the best model's curve is where we pick **one point** (the threshold) by cost: cost per missed churner (FN) vs cost per wasted offer (FP), within the team's capacity. *(The $270 vs $50 used in Notes 2 are **made-up numbers**; the real costs are set in phase 6, step T1.)*


#### 4. `"roc_auc"` vs `"average_precision"` in `cross_validate`: what each value in `cv_lr_df` is

Both scorers call **`predict_proba()`**, so neither uses the 0.5 threshold. In each fold, sklearn takes the validation customers' churn probabilities, tries **every** threshold, builds a curve, and turns the curve into one number. One number per fold gives the 5 rows; their mean is the value we report.

| | `"roc_auc"` → `test_roc_auc` | `"average_precision"` → `test_pr_auc` |
|---|---|---|
| Curve built per fold | ROC curve: TPR vs FPR | Precision–recall curve: precision vs recall |
| How the area is measured | **Exact area** under the ROC curve | **Staircase sum:** AP = Σ (Rₙ − Rₙ₋₁) × Pₙ. At each threshold: how much recall went up × the precision there, all added up |
| Is it "the AUC"? | Yes, it is the ROC-AUC | It is the standard way to report **PR-AUC**. Very close to the area under the PR curve, but uses rectangles (steps) instead of straight lines between points |
| Random model | 0.50 | 0.265 (= churn rate) |
| Our LR, mean of 5 folds | **0.846** | **0.663** |
| Business question it answers | "Does the model rank churners above loyal customers?" | "How clean is the call list as the team works down it?" |

**Why `average_precision` and not `auc(recall, precision)`?**
- `sklearn.metrics.auc(recall, precision)` joins the PR points with **straight lines** (trapezoids). On a PR curve that can **over-estimate** the area.
- `average_precision` uses **steps**, so it doesn't over-estimate. It is sklearn's recommended PR-AUC.
- The two numbers are usually very close; just say which one you report.

**Interview phrasing:** "I report PR-AUC as average precision, the step-wise area under the precision–recall curve, because trapezoidal interpolation can over-estimate it."


### Adding Training Score to CV

We also want the score on the **training folds**: customers the model **learned from**. That's like testing the student with the **same** questions they practised on.

Comparing the two tells you something important:

| Training score vs validation score | What it means | Student analogy |
| --- | --- | --- |
| About the same | The model learned real patterns ✅ | Understood the topic |
| Training much higher | The model **memorised** the training customers (overfitting) ❌ | Memorised the answers, fails on new questions |

This is leakage check **L6**. The spec says the gap should be **below about 0.05**.

In [14]:
cv_lr = cross_validate(pipe_lr, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1,
                       return_train_score=True)   # ← new: also score the training folds

cv_lr_df = pd.DataFrame(cv_lr)
cv_lr_df.agg(["mean", "std"]).T   # .T turns it so each metric is a row

,mean,std
fit_time,0.151512,0.007859
score_time,0.081885,0.008275
test_f1,0.590424,0.028894
train_f1,0.599183,0.005851
test_precision,0.654277,0.028507
train_precision,0.661003,0.007109
test_recall,0.539130,0.041799
train_recall,0.547993,0.007812
test_roc_auc,0.844776,0.012990
train_roc_auc,0.847790,0.003194



|  | Value |
| --- | --- |
| Training ROC-AUC (customers it learned from) | 0.852 |
| Validation ROC-AUC (customers it hasn't seen) | 0.847 |
| **Gap** | **0.005**, far below the 0.05 limit ✅ |

The gap is **0.852 - 0.847 = 0.005**.

### What it means

* The model scores **almost the same** on new customers as on the ones it learned from. In the student analogy, it **understood** the topic rather than memorising answers. **No overfitting.**


* That's expected for Logistic Regression. It's a simple model (one weight per feature), so it doesn't have enough "memory" to memorise 4,500 customers.


* Something to keep in mind for later: when train and validation are close **and** both are moderate, the model is limited by its **simplicity**, not by overfitting. That's why the spec goes on to try more flexible models (trees, boosting). They might find patterns LR can't, but they're also the ones that tend to overfit. Then this same gap check will matter.



**Interview phrasing:** "Logistic Regression had a train–CV ROC-AUC gap of 0.005, so it wasn't overfitting. Its limit is model simplicity, which is why I compared more flexible models next."

Creating function `run_cv` to be added in `Churn/`

In [15]:
def run_cv(pipe, name):
    """Run 5-fold CV on a pipeline; return one summary row for the comparison table."""
    # Define metrics relevant for imbalanced data
    scoring_metrics = {
        "f1": "f1",
        "precision": "precision",
        "recall": "recall",
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    }
    
    ## Run cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv = pd.DataFrame(cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1, return_train_score=True))

    ### build the summary as a dictionary for model
    row = {
        "model": name,
        "roc_auc": cv['test_roc_auc'].mean(),          # mean of the "test_roc_auc" column
        "roc_auc_std": cv['test_roc_auc'].std(),      # std of the same column
        "pr_auc": cv['test_pr_auc'].mean(),           # mean of "test_pr_auc"
        "pr_auc_std": cv['test_pr_auc'].std(),
        "precision": cv['test_precision'].mean(),        # mean of "test_precision"
        "recall": cv['test_recall'].mean(),           # mean of "test_recall"
        "f1": cv['test_f1'].mean(),               # mean of "test_f1"
        "gap_roc_auc": cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean(),      # mean of train_roc_auc minus mean of test_roc_auc
        "fit_time": cv['fit_time'].mean(),         # mean of "fit_time"
    }
    return row


In [16]:
lr_cv_data =run_cv(pipe_lr, "Logistic Regression")
lr_cv_data

{'model': 'Logistic Regression',
 'roc_auc': np.float64(0.8447763865246944),
 'roc_auc_std': np.float64(0.01298965933760855),
 'pr_auc': np.float64(0.6595637026842134),
 'pr_auc_std': np.float64(0.018447840297885446),
 'precision': np.float64(0.6542770784543676),
 'recall': np.float64(0.5391304347826086),
 'f1': np.float64(0.5904236057919692),
 'gap_roc_auc': np.float64(0.0030140609075326452),
 'fit_time': np.float64(0.148834228515625)}

## Model Comparison Dataframe

In [17]:
## This list will hold the results of cross-validation for different models. Each entry will be a dictionary containing the model name and its performance metrics.
results_cv = [run_cv(pipe_lr, "Logistic Regression")]

In [18]:
model_compare_df = pd.DataFrame(results_cv)
model_compare_df

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.01299,0.659564,0.018448,0.654277,0.53913,0.590424,0.003014,0.126164


### Data leakage Checks

##### L2 Leakage check

In [19]:
y_train.value_counts(normalize=True)   # check the churn rate in the training set

Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

In [20]:
y_test.value_counts(normalize=True)   # check the churn rate in the training set

Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64

I split first, stratified on churn, and confirmed train and test both have ≈ 26.5% churners. So the test set is untouched and representative, and CV and test scores are comparable.

#### L1: Target Leakage Review

In [21]:
df.loc[df['Churn']==1, ['tenure',  'MonthlyCharges', 'TotalCharges','Churn']].head(10)

,tenure,MonthlyCharges,TotalCharges,Churn
2,2,53.85,108.15,1
4,2,70.70,151.65,1
5,8,99.65,820.50,1
8,28,104.80,3046.05,1
13,49,103.70,5036.30,1
18,10,55.20,528.35,1
20,1,39.65,39.65,1
22,1,20.15,20.15,1
26,47,99.35,4749.15,1
27,1,30.20,30.20,1


* 🟩 **L1 passed.** Your conclusion is right: every feature is known at scoring time, so nothing leaks.



One thing to add for the **engineered** features. They're safe for two reasons:

1. They're **built only from known columns**: `tenure_group` $\leftarrow$ `tenure`, `num_services` $\leftarrow$ the 5 add-ons, `has_family` $\leftarrow$ Partner + Dependents.


2. They **don't learn from the data or the target**. The tenure bins are fixed edges from config, not quantiles, and none of them looks at `Churn`.



**Write it down in the notebook.** L1 says "record: the table in the notebook". A grouped table is enough, so you don't need 18 rows:

| Feature group | Columns | Known before churn? | Note |
| --- | --- | --- | --- |
| Account | tenure, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges | Yes | Current values available in billing at scoring time. Tenure is frozen at exit for churners (snapshot caveat, not leakage) |
| Demographics | SeniorCitizen, Partner, Dependents | Yes | Known at sign-up |
| Services | MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV | Yes | Current subscriptions |
| Engineered | tenure_group, num_services, has_family | Yes | Built only from the rows above; fixed rules, no target used |
| Excluded | customerID, TotalCharges, gender, PhoneService, StreamingMovies | n/a | See `config.DROP_COLS` |

So far, L1 and L2 have both passed.

### L3 Duplicates Check

In [22]:
train_profiles = X_train.drop(columns=config.ID_COL)
test_profiles  = X_test.drop(columns=config.ID_COL)

shared = pd.merge(train_profiles, test_profiles, how="inner").drop_duplicates()
## shared only has rows which are common to both the training and test sets. 
## This is a problem because it means that the model has already seen these rows during training, which can lead to data leakage and overfitting. 
## We need to ensure that the training and test sets are completely separate and do not share any rows.
len(shared)


12

22 feature profiles repeat across different customers; 12 of them span the train/test split. That's under 1% of test, they're genuine customers, so I kept them and recorded the count rather than deleting real data.

### L5 the single-feature AUC scan (the last check before modelling)

**The idea.** L1 checked **timing** by reasoning. L5 checks **strength** with numbers. For each feature on its own, train a tiny model that uses **only that one column** and get its CV ROC-AUC.

* $\text{AUC} \approx 0.5 \rightarrow$ this feature alone is useless.


* $\text{AUC} \approx 0.6\text{--}0.8 \rightarrow$ a normal, useful feature.


* **$\text{AUC} > 0.90$ from ONE column** $\rightarrow$ suspicious. Real churn is never that easy to predict from a single fact, so you investigate it as possible leakage.

**Bonus:** the ranking should match your EDA. Contract and tenure should be near the top, and that's a nice "my EDA and my model agree" moment.

In [23]:
len(config.NUM_COLS + config.CAT_COLS)

15

In [24]:
rows = []
for col in config.NUM_COLS + config.CAT_COLS:
    if col in config.NUM_COLS:
        pipe = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr", num_cols=[col], cat_cols=[])
    else:
        pipe = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr", num_cols=[], cat_cols=[col])

    cv = run_cv(pipe, "Logistic Regression")
    rows.append({"Column": col, "mean_roc_auc": cv["roc_auc"], "pr_auc": cv["pr_auc"]})

auc_scan = pd.DataFrame(rows).sort_values(by="mean_roc_auc", ascending=False)
auc_scan


,Column,mean_roc_auc,pr_auc
12,Contract,0.739499,0.415703
0,tenure,0.735243,0.478918
6,InternetService,0.697570,0.387264
14,PaymentMethod,0.676426,0.394965
1,MonthlyCharges,0.623485,0.316684
13,PaperlessBilling,0.610222,0.320356
7,OnlineSecurity,0.589535,0.306816
4,Dependents,0.586739,0.305367
10,TechSupport,0.585391,0.304694
3,Partner,0.582467,0.305129


Good question, and the answer is short. Your `run_cv` already returns **both**, so keep both columns in the table. ROC-AUC is the one we **sort and judge by** in this check, for one reason: **its scale is the same on every dataset.**

|  | Random-guess value | "Suspiciously high" |
| --- | --- | --- |
| **ROC-AUC** | **0.5** always, whatever the churn rate | $> 0.90$ is a widely used alarm level |
| **PR-AUC** | **= the churn rate** ($0.265$ here; $0.05$ on a $5\%$ churn dataset) | no universal number; it depends on the dataset |

L5 is an **alarm with a fixed rule** ("any single feature $> 0.90 \rightarrow$ investigate"). A fixed rule needs a metric whose "random" and "perfect" points don't move. ROC-AUC has that, and PR-AUC doesn't. A PR-AUC of $0.50$ means very different things at $26.5\%$ churn and at $5\%$ churn.

**When PR-AUC is the better choice:** comparing **models on the same dataset** for the business decision, because it focuses on how well we find the minority churners. That's why the spec uses it as the tie-breaker in step 10. Here, though, we're not choosing a model, just looking for a cheating feature.

**In practice:** sort by `roc_auc`, keep `pr_auc` alongside, and glance at whether both rank the same features at the top. They usually agree. If one feature were high on PR-AUC but ordinary on ROC-AUC, that would also be worth a look.

> As a leakage check I scanned each feature's standalone CV ROC-AUC. The max was Contract at 0.74, nowhere near the 0.90 alarm, and the ranking matched my EDA: contract and tenure on top.

## Step 2a: Logistic Regression, default vs `class_weight="balanced"`

**The problem it addresses.** Only 26.5% of customers churn. By default, LR treats every mistake the same. Since about 3 in 4 customers stay, the easiest way for it to reduce mistakes is to **lean towards "stays"**. That's why your default LR has a recall of only about 0.53: it misses almost half the churners.

**What `class_weight="balanced"` does.** It tells LR: *"a mistake on a churner counts more than a mistake on a stayer."* The weights come out roughly as:

```text
weight = n_samples / (2 × n_in_class)
stayers  (73.5%) -> ≈ 0.68
churners (26.5%) -> ≈ 1.89     <- each churner counts ~2.8x as much as a stayer
```

It's as if each churner were in the data about 2.8 times, but **without copying any rows**. That matters because the spec avoids SMOTE and other resampling in cycle 1.



In scikit-learn, the `class_weight` parameter in `LogisticRegression` is used to handle **imbalanced datasets** by altering the loss function to penalize misclassifications of minority classes more heavily.

The parameter can be configured in **three different ways**:

##### 1. `class_weight=None` (Default)

All classes are given **equal weight (1.0)**. The model optimizes for overall accuracy, which means it will naturally favor the majority class in heavily imbalanced data.

##### 2. `class_weight='balanced'`

Scikit-learn automatically adjusts the weights **inversely proportional to class frequencies** using the following heuristic formula:

$$w_j = \frac{n_{\text{samples}}}{n_{\text{classes}} \cdot n_j}$$

* Where $w_j$ is the weight assigned to class $j$, $n_{\text{samples}}$ is the total number of samples, $n_{\text{classes}}$ is the total number of unique classes, and $n_j$ is the count of samples in class $j$.

---

##### 3. Custom Weights via a Dictionary

You can manually define specific weights for each class label by passing a dictionary in the format `{class_label: weight}`. This is ideal when you want to explicitly punish false negatives or false positives based on custom business costs.

**Code Example:**

```python
from sklearn.linear_model import LogisticRegression

# Giving the minority class (1) five times the weight of the majority class (0)
custom_weights = {0: 1.0, 1: 5.0}

model = LogisticRegression(class_weight=custom_weights)
model.fit(X_train, y_train)

```

---

##### How it Works Internally

Setting `class_weight` **does not** oversample or duplicate your data rows. Instead, it applies a weight multiplier directly to the negative log-likelihood (loss function) calculation for each individual sample during training. Errors made on the heavily weighted minority class result in a larger gradient step, forcing the optimization algorithm to adjust the decision boundary to better accommodate the minority class.

##### Performance Trade-off

While adjusting class weights typically improves performance metrics like **recall** (by reducing false negatives for the minority class), it can simultaneously lower your **precision**, resulting in a higher number of false positives.

In [25]:
pipe_lr_bal = build_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced"), model_type="lr")
results_lr_bal = run_cv(pipe_lr_bal, "Logistic Regression (balanced)")
results_lr_bal

{'model': 'Logistic Regression (balanced)',
 'roc_auc': np.float64(0.8447651528769539),
 'roc_auc_std': np.float64(0.012848927810704867),
 'pr_auc': np.float64(0.6586644856752322),
 'pr_auc_std': np.float64(0.01837776928452644),
 'precision': np.float64(0.5182980161220382),
 'recall': np.float64(0.7959866220735786),
 'f1': np.float64(0.6276056363442946),
 'gap_roc_auc': np.float64(0.0029253412048911853),
 'fit_time': np.float64(0.11989946365356445)}

In [26]:
results_cv.append(run_cv(pipe_lr_bal, "Logistic Regression (balanced)"))

In [27]:
results_cv

[{'model': 'Logistic Regression',
  'roc_auc': np.float64(0.8447763865246944),
  'roc_auc_std': np.float64(0.01298965933760855),
  'pr_auc': np.float64(0.6595637026842134),
  'pr_auc_std': np.float64(0.018447840297885446),
  'precision': np.float64(0.6542770784543676),
  'recall': np.float64(0.5391304347826086),
  'f1': np.float64(0.5904236057919692),
  'gap_roc_auc': np.float64(0.0030140609075326452),
  'fit_time': np.float64(0.12616362571716308)},
 {'model': 'Logistic Regression (balanced)',
  'roc_auc': np.float64(0.8447651528769539),
  'roc_auc_std': np.float64(0.012848927810704867),
  'pr_auc': np.float64(0.6586644856752322),
  'pr_auc_std': np.float64(0.01837776928452644),
  'precision': np.float64(0.5182980161220382),
  'recall': np.float64(0.7959866220735786),
  'f1': np.float64(0.6276056363442946),
  'gap_roc_auc': np.float64(0.0029253412048911853),
  'fit_time': np.float64(0.12684235572814942)}]

In [28]:
model_compare_df = pd.DataFrame(results_cv)
model_compare_df 

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.126164
1,Logistic Regression (balanced),0.844765,0.012849,0.658664,0.018378,0.518298,0.795987,0.627606,0.002925,0.126842


#### weighting moves the cut-off, not the ranking



```text
Default LR:
Score ranking
0.82  churner  ┐
0.61  churner  | flagged
0.55  stayer   ┘
─────── 0.5 ───────
0.38  churner
0.20  stayer
```

```text
Balanced LR:
Score ranking (same order, everyone shifted up)
0.93  churner  ┐
0.81  churner  |
0.76  stayer   | flagged
0.68  churner  |
0.59  stayer   ┘
─────── 0.5 ───────
0.36  stayer
```

- Customers stay in **almost the same order**, so the model separates churners from stayers equally well. That's why **ROC-AUC and PR-AUC don't change**.
- Every score is pushed up, so **more customers cross 0.5**. That's why **recall rises and precision falls**.

So `class_weight="balanced"` **does almost the same thing as keeping the default model and lowering the threshold**. It doesn't make LR any smarter.

---

#### In business terms (per CV fold: about 1,127 customers, 299 churners)

| | Churners caught | Customers flagged | Wasted offers |
| :--- | :--- | :--- | :--- |
| Default (0.5) | $0.537 \times 299 \approx \mathbf{161}$ | $161 / 0.675 \approx \mathbf{238}$ | $\approx 77$ |
| Balanced (0.5) | $0.796 \times 299 \approx \mathbf{238}$ | $238 / 0.522 \approx \mathbf{456}$ | $\approx 218$ |

Balanced catches 77 more churners but nearly **doubles** the retention team's call list. Which one is better depends on the offer cost and the team's capacity. That's exactly the phase 6 threshold decision, not a model-quality question.

#### So what's the decision for LR?

Weighting gives **no gain in ranking**. It also has a downside: the probabilities stop being honest. A customer scored 0.6 by the balanced model doesn't really have a 60% chance of churning. Phase 6 uses those probabilities for the cost-based threshold, and the spec notes this risk in step 8.

My recommendation: **LR, class weighting off**. Keep the default model and get the higher recall from the threshold in phase 6. 


### Step 2b: feature question 1, TotalCharges in or out?

In [29]:
pipe_lr = build_pipeline(LogisticRegression(max_iter=1000),  model_type="lr", num_cols=config.NUM_COLS + ['TotalCharges'], cat_cols=config.CAT_COLS)
results_lr_bal_variant = run_cv(pipe_lr, "Logistic Regression - TotalCharges included")

In [30]:
pd.DataFrame(results_cv + [results_lr_bal_variant])

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.126164
1,Logistic Regression (balanced),0.844765,0.012849,0.658664,0.018378,0.518298,0.795987,0.627606,0.002925,0.126842
2,Logistic Regression - TotalCharges included,0.846057,0.013601,0.661824,0.018758,0.655259,0.544482,0.593977,0.003215,0.171087


> `TotalCharges` is roughly tenure × MonthlyCharges. Adding it to LR slightly lowered CV ROC-AUC (0.847341), consistent with collinearity adding noise and no new signal, so I left it out.

### Step 2c: feature question 2, `has_family`

In [31]:
pipe_lr_fam = build_pipeline(LogisticRegression(max_iter=1000),  model_type="lr", num_cols=config.NUM_COLS, cat_cols=[col  for col in config.CAT_COLS if col != 'has_family'])
results_lr_fam_variant = run_cv(pipe_lr_fam, "Logistic Regression - has_family excluded")

In [32]:
pd.DataFrame(results_cv + [results_lr_fam_variant])

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.126164
1,Logistic Regression (balanced),0.844765,0.012849,0.658664,0.018378,0.518298,0.795987,0.627606,0.002925,0.126842
2,Logistic Regression - has_family excluded,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.131059


> `has_family` is just Partner OR Dependents, both already in the model. Removing it didn't change CV ROC-AUC beyond fold noise, so I kept the simpler feature set.

### `tenure_group`


In [33]:
pipe_lr_fam = build_pipeline(LogisticRegression(max_iter=1000),  model_type="lr", num_cols=config.NUM_COLS, cat_cols=[col  for col in config.CAT_COLS if col != 'tenure_group'])
results_lr_tenure_grp_variant = run_cv(pipe_lr_fam, "Logistic Regression - tenure_group excluded")

In [34]:
pd.DataFrame(results_cv + [results_lr_tenure_grp_variant])

,model,roc_auc,roc_auc_std,pr_auc,pr_auc_std,precision,recall,f1,gap_roc_auc,fit_time
0,Logistic Regression,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.126164
1,Logistic Regression (balanced),0.844765,0.012849,0.658664,0.018378,0.518298,0.795987,0.627606,0.002925,0.126842
2,Logistic Regression - tenure_group excluded,0.844776,0.012990,0.659564,0.018448,0.654277,0.539130,0.590424,0.003014,0.134600


* The difference between Baslne ROC-AUC  and Logistic regression withou `tenure_group` is $\mathbf{0.847386 - 0.844328 = 0.0031}$, in favour of keeping `tenure_group`.

That's **smaller than the fold std ($\mathbf{\sim0.01}$)**, so the simple rule would say "noise, remove it". But the theory says the bands should help LR. When the two disagree, the **fold-by-fold check** settles it. As done in below cell:

**What's new here:**

- `cross_val_score` is the little sibling of `cross_validate`. It computes **one** metric and returns just the 5 per-fold values as an array. That's all we need.
- The **same** `skf` with the same seed means fold 1 holds the same customers for both models. That's what makes a fold-by-fold comparison fair.

In [35]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=config.N_SPLITS, shuffle=True, random_state=config.RANDOM_STATE)

cat_no_tg = [c for c in config.CAT_COLS if c != "tenure_group"]
pipe_no_tg = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr", cat_cols=cat_no_tg)

auc_base  = cross_val_score(pipe_lr,    X_train, y_train, cv=skf, scoring="roc_auc")
auc_no_tg = cross_val_score(pipe_no_tg, X_train, y_train, cv=skf, scoring="roc_auc")

auc_base - auc_no_tg     # 5 numbers, one per fold


array([0.00050894, 0.00039382, 0.00112694, 0.00164801, 0.00272573])

**3 of 5 folds are positive and 2 are negative.** That's the "mixed" row of the table, so the honest verdict is **noise $\rightarrow$ remove `tenure_group**`.

I expected it to help, and the data says it doesn't, clearly enough. **We follow the rule we set *before* seeing the numbers.** That's important: if you change the rule after seeing the result, you can talk yourself into anything. This is how you avoid it, and it's a great interview point.

**Why the bands don't help even though the kink is real**

The kink is real, but other features **already capture it**:

* New customers are mostly on **month-to-month contracts** (`Contract` is in the model, and it's the top L5 feature).


* New customers often pay by **electronic cheque** and have **fibre**, which are also in the model.



So when LR sees "month-to-month + electronic cheque + low tenure", it already knows "high risk". The bands repeat what those columns say. The same pattern ran through TotalCharges and has_family: **in a model with many features, a feature has to add something the others don't.**

Also look at fold 5 (+0.009). That single fold provides most of the 0.003 average. This is why we check each fold and not just the mean.

**Interview sentence:**

> "I expected tenure bands to help LR capture the early-tenure risk, but fold-by-fold the gain was inconsistent (3 of 5 folds), because Contract and payment method already encode it. So I dropped it."

---

##### Score so far:

| Feature | Decision | Evidence |
| --- | --- | --- |
| TotalCharges | out | ROC-AUC fell when added |
| has_family | out | No change; repeats Partner + Dependents |
| tenure_group | out | +0.003, mixed folds (3/5) |

### `num_services` vs the 5 add-on columns

In [36]:
num_no_count = [c for c in config.NUM_COLS if c != "num_services"]            # drop the count and keep the other 55 add-on columns
cat_no_addons = [c for c in config.CAT_COLS if c not in config.ADDON_COLS]    # drop the 5 add-on columns

pipe_A = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr", num_cols=num_no_count)   # A: 5 columns only
pipe_B = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr", cat_cols=cat_no_addons)  # B: count only


In [37]:
auc_base = cross_val_score(pipe_lr, X_train, y_train, cv=skf, scoring="roc_auc")
auc_A    = cross_val_score(pipe_A,  X_train, y_train, cv=skf, scoring="roc_auc")
auc_B    = cross_val_score(pipe_B,  X_train, y_train, cv=skf, scoring="roc_auc")

print("base:", auc_base.mean(), " A:", auc_A.mean(), " B:", auc_B.mean())
print("A - B per fold:", auc_A - auc_B)

base: 0.8460570765816913  A: 0.8447763865246944  B: 0.8417576333271344
A - B per fold: [-0.00040796  0.00357068  0.00689092  0.00405741  0.00098272]


* **Result: A (the 5 columns) is slightly ahead of B (the count), in 4 of 5 folds.** But the gap is tiny, about **0.0007**.

##### Reading it

|  | ROC-AUC | vs base |
| --- | --- | --- |
| base (both) | 0.84734 | n/a |
| A: 5 columns | **0.84738** | the same |
| B: count only | 0.84666 | $-0.0007$ |

* **Base $\approx$ A.** Removing `num_services` costs nothing. The count is just the sum of the 5 columns, so LR gets no new information from it.

* **A vs B:** 4 of 5 folds positive, so it's consistent but **very small**. On numbers alone it's close to a tie.

##### When numbers tie, decide on something else

**A wins on interpretability:**

* **B** tells the business: *"each extra add-on lowers churn risk by X."*

* **A** tells the business: "* **TechSupport** and **OnlineSecurity** lower churn risk a lot; **StreamingTV** barely matters.*" That's **actionable**: the retention team knows *which* add-on to offer.

Your EDA already hinted at this, and L5 agrees: OnlineSecurity (0.590) and TechSupport (0.585) were stronger alone than StreamingTV (0.540) or DeviceProtection (0.533). The 5 services are **not** equal, and a count hides that.

**"Simpler" doesn't favour B as much as it looks.** To compute `num_services`, the app has to ask all 5 add-on questions anyway. B saves 4 coefficients, not 4 questions.

$\rightarrow$ **Decision: A, the 5 add-on columns. Drop `num_services` for LR.**

**Interview sentence:**

> "num_services is the exact sum of the five add-on flags, so LR should get one or the other. The binaries scored slightly higher in 4 of 5 folds, and they tell the business which add-on reduces churn, which a count can't."
> 
> 

(Trees may still find the count useful, since it's a ready-made split. We can retest that in step 5 or 6 by passing a different `num_cols`; config is just the default.)

#### Step B: the final combined check
All 4 decisions applied together vs the original baseline:

In [38]:
num_final = ["tenure", "MonthlyCharges"]                                      # TotalCharges out, num_services out
cat_final = [c for c in config.CAT_COLS if c not in ("has_family", "tenure_group")]

pipe_final = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr",
                            num_cols=num_final, cat_cols=cat_final)
auc_final = cross_val_score(pipe_final, X_train, y_train, cv=skf, scoring="roc_auc")

print("base:", auc_base.mean(), " final:", auc_final.mean())
print("final - base per fold:", auc_final - auc_base)


base: 0.8460570765816913  final: 0.8447763865246944
final - base per fold: [-0.00050894 -0.00039382 -0.00112694 -0.00164801 -0.00272573]


**Pass if**: final is about equal to base or better, with mixed or positive signs. That would mean 4 fewer features, same performance: a simpler model at no cost.

* **Result: final $-$ base $= -0.0026$, with 2 of 5 folds positive and 3 negative.** Mixed signs, smaller than the fold std, so by our rule **it passes: same performance, 4 fewer features.**


**Being honest about it.** The average did dip a little, and fold 5 ($-0.009$) is the same fold where `tenure_group` helped ($+0.009$) in its own test. So most of this dip is the bands' small, inconsistent effect showing up again. That's not a new problem; it's the same noise we already judged. In the interview, say it straight:

> "The reduced feature set scored 0.845 vs 0.847, inside fold-to-fold noise with mixed signs, so I chose the simpler model: 4 fewer features, easier to explain, fewer inputs to maintain."
> 
> 

Admitting the small dip and explaining why you accept it sounds far stronger than claiming "no change".

## Step 2c: reading LR's coefficients as odds ratios

**Why do this?** LR is the **interpretable** model. Here we check that what it learned **makes business sense** and **agrees with your EDA**. If the model said *"two-year contracts increase churn"*, something would be broken. It's also the best interview material in this phase, because you can say *"a two-year contract multiplies the odds of churn by 0.2"*.

**The idea, step by step:**

1. LR doesn't predict churn directly. It predicts the **log-odds** of churn:



$$\text{log-odds} = \text{intercept} + \text{coef}_1 \cdot \text{feature}_1 + \text{coef}_2 \cdot \text{feature}_2 + \dots$$

2. A coefficient on its own (e.g. $-1.4$) is hard to explain to a manager. `np.exp(coef)` **turns it into an odds ratio**:



| Odds ratio | Meaning |
| --- | --- |
| **1.0** | No effect |
| **$> 1$** | Raises the odds of churn ($2.0 =$ doubles them) |
| **$< 1$** | Lowers the odds of churn ($0.5 =$ halves them) |

3. Odds aren't the same as probability: $\text{odds} = p / (1 - p)$. At 26.5% churn the odds are $0.265 / 0.735 \approx 0.36$. "Doubling the odds" means $0.36 \rightarrow 0.72$, which is a probability of about 42%, not 53%.



**Two things that change how you read them in your pipeline:**

* **The numeric features are scaled**, so the odds ratio for `tenure` means *"per 1 standard deviation more tenure"* (about 24 months), not per month.


* For categories like `Contract`, compare **levels within the same feature**. "Two year vs Month-to-month" is the meaningful statement.

In [40]:
pipe_lr = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr").fit(X_train, y_train)

In [41]:
pipe_lr

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('collapse', ...), ('features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['customerID','gender','SeniorCitizen',...,'PaymentMethod', 'MonthlyCharges','TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function col...001F834567CE0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False


In [45]:
pipe_lr.named_steps["model"].coef_[0]

array([-0.79608778,  0.62883781,  0.14121536,  0.02324176, -0.23863145,
       -0.60181008,  0.28568643, -0.3776348 , -0.25913561,  0.14390966,
       -0.5785325 , -0.49582104, -0.23531552, -0.09709256, -0.44008091,
        0.0937522 ,  0.44587584, -0.25016906, -0.88946523,  0.36898595,
       -0.28830195, -0.32336469,  0.09974871, -0.18184052])

In [46]:
coefs = pipe_lr.named_steps["model"].coef_[0]
names = pipe_lr.named_steps["prep"].get_feature_names_out()

odds = (pd.DataFrame({"feature": names, "coef": coefs})
          .assign(odds_ratio=lambda d: np.exp(d["coef"]))
          .sort_values("odds_ratio"))
odds


,feature,coef,odds_ratio
18,cat__Contract_Two year,-0.889465,0.410875
0,num__tenure,-0.796088,0.451090
5,cat__MultipleLines_No,-0.601810,0.547819
10,cat__InternetService_No,-0.578533,0.560721
11,cat__OnlineSecurity_Yes,-0.495821,0.609071
14,cat__TechSupport_Yes,-0.440081,0.643984
7,cat__MultipleLines_Yes,-0.377635,0.685481
21,cat__PaymentMethod_Credit card (automatic),-0.323365,0.723710
20,cat__PaymentMethod_Bank transfer (automatic),-0.288302,0.749535
8,cat__InternetService_DSL,-0.259136,0.771718


Here's that code, piece by piece.

## Line 1: `coefs = pipe_lr.named_steps["model"].coef_[0]`

Read it left to right, like opening nested boxes:

| Piece | What it is |
|---|---|
| `pipe_lr` | Your whole Pipeline: a box holding 4 steps |
| `.named_steps` | A **dictionary** of those steps, keyed by the names you gave them: `{"collapse": ..., "features": ..., "prep": ..., "model": ...}` |
| `["model"]` | Takes out the step named `"model"`, your fitted `LogisticRegression` |
| `.coef_` | The **coefficients** LR learned, one number per input column. The `_` at the end is sklearn's way of saying "this only exists **after** `.fit()`" |
| `[0]` | `coef_` is shaped like a table with **1 row** × N columns, e.g. `[[-0.8, 0.3, ...]]`. `[0]` takes that row, giving a flat list `[-0.8, 0.3, ...]` |

*Why only 1 row?* sklearn keeps one row of coefficients per class it has to separate. With 2 classes (churn / stay), one set is enough, because the "stay" score is just the opposite of the "churn" score.

## Line 2: `names = pipe_lr.named_steps["prep"].get_feature_names_out()`

| Piece | What it is |
|---|---|
| `named_steps["prep"]` | Takes out the fitted **ColumnTransformer** (the scaler + one-hot encoder) |
| `.get_feature_names_out()` | A **method** (note the `()`, so it's a function you call). It returns the names of the columns that `prep` **outputs**, the ones the model actually sees |

This step is needed because one-hot encoding changes the columns:
```
Input to prep:   Contract  (1 column, 3 values)
Output of prep:  cat__Contract_Month-to-month, cat__Contract_One year, cat__Contract_Two year
```
The model learns **one coefficient per output column**, so you need the output names to label them.

**Why the two lists line up:** `prep` hands its columns to `model` in a fixed order. Coefficient #5 belongs to column #5. So position 0 in `names` matches position 0 in `coefs`, and so on.

## The DataFrame step

```python
odds = (pd.DataFrame({"feature": names, "coef": coefs})
          .assign(odds_ratio=lambda d: np.exp(d["coef"]))
          .sort_values("odds_ratio"))
```

**The outer `( ... )`** lets you split one long statement over several lines. Python reads it all as **one expression**.

**This style is called method chaining:** each line takes the result of the line above and does one more thing to it. Read it like a recipe.

**Step 1: `pd.DataFrame({"feature": names, "coef": coefs})`**
Builds a table from a dictionary. Each key becomes a column name, and each list fills that column:
```
   feature                  coef
0  num__tenure             -0.85
1  num__MonthlyCharges      0.40
2  cat__Contract_Two year  -1.30
...
```

**Step 2: `.assign(odds_ratio=lambda d: np.exp(d["coef"]))`**
Adds a new column called `odds_ratio`.
- `lambda d: ...` is a **tiny one-line function with no name**. In plain words: *"take the table (call it `d`) and return `np.exp` of its `coef` column."*
- Why the lambda and not just `odds["coef"]`? Inside a chain, the table **doesn't have a variable name yet**. `odds` only exists once the whole chain finishes. The lambda gets handed "the table so far".
- `np.exp(...)` computes *e* to the power of each coefficient. That turns log-odds into odds ratios: −1.30 → 0.27, 0 → 1.0, 0.40 → 1.49.

**Step 3: `.sort_values("odds_ratio")`**
Sorts rows from the smallest odds ratio to the largest:
- **Top of the table:** odds ratio < 1, protects against churn
- **Bottom of the table:** odds ratio > 1, raises churn risk

**The same thing without chaining**, if that's easier to read:
```python
odds = pd.DataFrame({"feature": names, "coef": coefs})
odds["odds_ratio"] = np.exp(odds["coef"])
odds = odds.sort_values("odds_ratio")
```
The result is identical. Use whichever you can explain more comfortably.


**Your protective picks are right ✅:** `Contract_Two year` (0.41) and `tenure` (0.45).

## The strongest risk factor: `MonthlyCharges` (odds ratio 1.88)

Since `MonthlyCharges` is scaled, this means: **each 1 standard deviation higher monthly bill (about $30) multiplies the odds of churn by ~1.9**, with the other features held the same. Next come `Contract_Month-to-month` (1.56) and `PaperlessBilling_Yes` (1.45).

## Reading categories properly: compare levels within a feature

Each odds ratio in the table is relative to a hidden "average" level, so the cleaner statement compares **two levels of the same feature**. Subtract their coefficients, then take `exp`:

| Comparison | Calculation | Odds ratio | In plain words |
|---|---|---|---|
| Two-year vs month-to-month contract | exp(−0.889 − 0.446) | **0.26** | A two-year contract has about **¼ the odds** of churn |
| One-year vs month-to-month contract | exp(−0.250 − 0.446) | **0.50** | Half the odds |
| Fiber vs no internet | exp(0.144 + 0.579) | **2.06** | Fiber customers have about **2× the odds** |
| Fiber vs DSL | exp(0.144 + 0.259) | **1.50** | 1.5× the odds |
| Electronic check vs credit card (auto) | exp(0.100 + 0.323) | **1.53** | Manual payers have 1.5× the odds |

**This matches your EDA:** contract type is the biggest lever, new customers are the risk, fiber customers churn more, electronic cheque payers churn more, and TechSupport and OnlineSecurity protect. ✅ The model learned the same story your EDA told.

**One honest caveat (good for interviews):** each odds ratio means *"holding all other features fixed"*. `MonthlyCharges` is largely **set by** the services a customer has (fiber + add-ons = a higher bill). So LR splits the credit between the bill and the services, which is why some service coefficients look odd. For example, `MultipleLines_Yes` (0.69) looks protective: *at the same bill*, having that service means paying less for everything else. Read single coefficients of correlated features with caution. The **Contract, tenure and payment** results are the robust ones.

---

## Where does `get_feature_names_out` come from?

**You didn't create it, and neither did your project.** It's a method **built into scikit-learn**. Every sklearn transformer has one: `StandardScaler`, `OneHotEncoder`, `ColumnTransformer`, and so on.

**How it works for your `prep` step:**
```
ColumnTransformer.get_feature_names_out()
   ├─ asks StandardScaler  → ["tenure", "MonthlyCharges"]           → adds "num__"
   └─ asks OneHotEncoder   → ["Contract_Two year", ...]             → adds "cat__"
```
The `num__` / `cat__` prefixes are the names **you** gave the branches in [features.py:109-110](churn/features.py#L109-L110).

**To see where it's defined:** in VS Code, **Ctrl+click** `get_feature_names_out` in the notebook. That opens sklearn's own source file, `sklearn/compose/_column_transformer.py`, inside `.venv`. Or run `help(pipe_lr.named_steps["prep"].get_feature_names_out)`.

Same idea as `.fit()` or `.coef_`: they come with the library, and you just call them. Your own functions are the ones in `churn/` (like `build_preprocessor`).
